# Data Importation

In [0]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from pyspark.sql.functions import col, lit, to_timestamp, current_timestamp
import pyspark.sql.functions as F
from pyspark.sql.types import * 
from delta import DeltaTable

At this stage, we convert raw files from S3 into Delta tables in the Bronze layer, preserving incremental ingestion and enabling controlled reprocessing.

The ingestion strategy now supports **two execution modes**:
- **Rolling mode (`RUN_MODE = "rolling"`)**: applies an automatic moving window based on `ANALYSIS_WINDOW_MONTHS`, processing only recent data for regular operations.
- **Backfill mode (`RUN_MODE = "backfill"`)**: processes a user-defined interval (`BACKFILL_START` to `BACKFILL_END`) for historical reprocessing without full reloads.

To keep this robust and auditable, each table has its own configuration in `tables_config`, including:
- source path
- target Delta table
- file format (JSON/Avro)
- temporal column used for filtering (`time_col`)

> Note: some datasets do not have a business event timestamp in this ingestion step (for example, registry/profile datasets). In those cases, `time_col = None`, and no temporal filter is applied at Bronze ingestion.

### Incremental Guarantees and Backfill Safety
- **Auto Loader (`cloudFiles`)** detects newly landed files efficiently.
- **Append mode** avoids destructive overwrites.
- **Checkpoint and schema locations are separated by mode/window**, preventing conflicts between daily rolling ingestion and historical backfills.
- **`trigger(availableNow=True)`** processes the currently available backlog and then stops, which is suitable for scheduled batch-like execution.

This design keeps the complete historical source of truth in S3 while letting Bronze process only the required temporal slice for analytics and ML workflows.

In [0]:

# Data source paths
SOURCE_PATH_1 = "s3://aurorapay-studycase/customer_profiles"
SOURCE_PATH_2 = "s3://aurorapay-studycase/device_signals"
SOURCE_PATH_3 = "s3://aurorapay-studycase/merchant_registry"
SOURCE_PATH_4 = "s3://aurorapay-studycase/security_logs"
SOURCE_PATH_5 = "s3://aurorapay-studycase/transaction_events"

# Target tables where data will be stored
TARGET_TABLE_1 = "fraud_detection_project.bronze_layer.customer_profiles"
TARGET_TABLE_2 = "fraud_detection_project.bronze_layer.device_signals"
TARGET_TABLE_3 = "fraud_detection_project.bronze_layer.merchant_registry"
TARGET_TABLE_4 = "fraud_detection_project.bronze_layer.security_logs"
TARGET_TABLE_5 = "fraud_detection_project.bronze_layer.transaction_events"
# -----------------------------------------

# Execution Mode
RUN_MODE = "rolling"  # "rolling" or "backfill"

ANALYSIS_WINDOW_MONTHS = 6
BACKFILL_START = "2025-10-01"   # used only in backfill mode
BACKFILL_END   = "2025-12-31"

tables_config = { # it is a dictionary with key and value
    SOURCE_PATH_1: {"target": TARGET_TABLE_1, "format": "json", "time_col": None},
    SOURCE_PATH_2: {"target": TARGET_TABLE_2, "format": "avro", "time_col": "ingestion_timestamp"},
    SOURCE_PATH_3: {"target": TARGET_TABLE_3, "format": "json", "time_col": None},
    SOURCE_PATH_4: {"target": TARGET_TABLE_4, "format": "avro", "time_col": "ingestion_timestamp"},
    SOURCE_PATH_5: {"target": TARGET_TABLE_5, "format": "avro", "time_col": "txn_timestamp"},
}

# Automatic calculation of the cutoff date
# If the job runs in 3 weeks, this date advances together, keeping the moving window
cutoff_date = datetime.now() - timedelta(days=ANALYSIS_WINDOW_MONTHS * 30)
cutoff_filter_str = cutoff_date.strftime("%Y-%m-%d")

print(f"--- Start of Inference Cycle ---")
print(f"Execution Base Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"Configured Window: Last {ANALYSIS_WINDOW_MONTHS} months")
print(f"Calculated Date Filter: >= {cutoff_filter_str}")
print(f"Note: The ingestion below brings all new data (Incremental). The filter should be used when reading these tables.")
# -----------------------------------------

CHECKPOINT_BASE_PATH = "s3://aurorapay-studycase/checkpoints"


def build_time_filter(df, time_col, run_mode, months, start_date, end_date):
    if not time_col:
        return df  # without temporal filter by column (dimensions/lookups)

    ts_col = F.to_timestamp(F.col(time_col))

    if run_mode == "rolling":
        cutoff = F.add_months(F.current_timestamp(), -months)
        return df.filter(ts_col >= cutoff)

    if run_mode == "backfill":
        start_ts = F.to_timestamp(F.lit(start_date))
        end_ts = F.to_timestamp(F.lit(end_date + " 23:59:59"))
        return df.filter((ts_col >= start_ts) & (ts_col <= end_ts))

    raise ValueError(f"RUN_MODE invalido: {run_mode}")


for source, cfg in tables_config.items(): 
    target = cfg["target"] # cfg is associated to the value of the dictionary
    file_format = cfg["format"]
    time_col = cfg["time_col"]
    table_name = target.split(".")[-1] # will take only the 
    
    # Checkpoint separated by mode/window
    if RUN_MODE == "rolling":
        ckpt = f"{CHECKPOINT_BASE_PATH}/rolling/{table_name}"
        schema_loc = f"{CHECKPOINT_BASE_PATH}/rolling/{table_name}_schema"
    else:
        window_id = f"{BACKFILL_START}_{BACKFILL_END}".replace("-", "")
        ckpt = f"{CHECKPOINT_BASE_PATH}/backfill/{table_name}/{window_id}"
        schema_loc = f"{CHECKPOINT_BASE_PATH}/backfill/{table_name}/{window_id}_schema"

    df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", file_format)
          .option("cloudFiles.inferColumnTypes", "true")
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .option("cloudFiles.schemaLocation", schema_loc)
          .option("maxFilesPerTrigger", 1000)
          .load(source)
          .select("*", "_metadata.file_path")
          .withColumn("ingest_datetime", F.current_timestamp()))

    df = build_time_filter(df, time_col, RUN_MODE, ANALYSIS_WINDOW_MONTHS, BACKFILL_START, BACKFILL_END)

    (df.writeStream
       .format("delta")
       .outputMode("append")
       .option("checkpointLocation", ckpt)
       .option("mergeSchema", "true")
       .trigger(availableNow=True)
       .toTable(target))